In [1]:
import polars as pl                            # Import required libraries.

In [2]:
sales = pl.read_csv(r"../data/raw/Sales.csv")
# Read the first dataset.

In [3]:
sales.glimpse()

Rows: 62884
Columns: 9
$ Order Number  <i64> 366000, 366001, 366001, 366002, 366002, 366002, 366004, 366004, 366005, 366007
$ Line Item     <i64> 1, 1, 2, 1, 2, 3, 1, 2, 1, 1
$ Order Date    <str> '1/1/2016', '1/1/2016', '1/1/2016', '1/1/2016', '1/1/2016', '1/1/2016', '1/1/2016', '1/1/2016', '1/1/2016', '1/1/2016'
$ Delivery Date <str> null, '1/13/2016', '1/13/2016', '1/12/2016', '1/12/2016', '1/12/2016', null, null, null, null
$ CustomerKey   <i64> 265598, 1269051, 1269051, 266019, 266019, 266019, 1107461, 1107461, 844003, 2035771
$ StoreKey      <i64> 10, 0, 0, 0, 0, 0, 38, 38, 33, 43
$ ProductKey    <i64> 1304, 1048, 2007, 1106, 373, 1080, 163, 1529, 421, 1617
$ Quantity      <i64> 1, 2, 1, 7, 1, 4, 6, 2, 4, 1
$ Currency Code <str> 'CAD', 'USD', 'USD', 'CAD', 'CAD', 'CAD', 'GBP', 'GBP', 'EUR', 'USD'



In [4]:
total_null = sum(sales.null_count())[0]
total_rows = len(sales)
null_percentage = round(total_null/total_rows * 100,2)
print(f"Null percentage: {null_percentage} %")
# Only Delivery Date column has null values, which makes up approximately 80 % of total values.

Null percentage: 79.06 %


In [5]:
sales.is_duplicated().sum()
# No duplicate rows were found in the dataset.

0

In [6]:
sales.head(20)

Order Number,Line Item,Order Date,Delivery Date,CustomerKey,StoreKey,ProductKey,Quantity,Currency Code
i64,i64,str,str,i64,i64,i64,i64,str
366000,1,"""1/1/2016""",null,265598,10,1304,1,"""CAD"""
366001,1,"""1/1/2016""","""1/13/2016""",1269051,0,1048,2,"""USD"""
366001,2,"""1/1/2016""","""1/13/2016""",1269051,0,2007,1,"""USD"""
366002,1,"""1/1/2016""","""1/12/2016""",266019,0,1106,7,"""CAD"""
366002,2,"""1/1/2016""","""1/12/2016""",266019,0,373,1,"""CAD"""
…,…,…,…,…,…,…,…,…
366010,1,"""1/1/2016""","""1/8/2016""",370077,0,618,5,"""CAD"""
366011,1,"""1/1/2016""",null,1984985,66,128,7,"""USD"""
366011,2,"""1/1/2016""",null,1984985,66,1638,1,"""USD"""


In [7]:
sales = sales.with_columns(
    pl.col("Order Date").str.to_date("%m/%d/%Y").alias("Order_Date"),
    pl.col("Delivery Date").str.to_date("%m/%d/%Y").alias("Delivery_Date")
)
sales
# Changed the type of Order Date/Delivery Date columns from str to date.

Order Number,Line Item,Order Date,Delivery Date,CustomerKey,StoreKey,ProductKey,Quantity,Currency Code,Order_Date,Delivery_Date
i64,i64,str,str,i64,i64,i64,i64,str,date,date
366000,1,"""1/1/2016""",null,265598,10,1304,1,"""CAD""",2016-01-01,null
366001,1,"""1/1/2016""","""1/13/2016""",1269051,0,1048,2,"""USD""",2016-01-01,2016-01-13
366001,2,"""1/1/2016""","""1/13/2016""",1269051,0,2007,1,"""USD""",2016-01-01,2016-01-13
366002,1,"""1/1/2016""","""1/12/2016""",266019,0,1106,7,"""CAD""",2016-01-01,2016-01-12
366002,2,"""1/1/2016""","""1/12/2016""",266019,0,373,1,"""CAD""",2016-01-01,2016-01-12
…,…,…,…,…,…,…,…,…,…,…
2243030,1,"""2/20/2021""",null,1216913,43,632,3,"""USD""",2021-02-20,null
2243031,1,"""2/20/2021""","""2/24/2021""",511229,0,98,4,"""EUR""",2021-02-20,2021-02-24
2243032,1,"""2/20/2021""","""2/23/2021""",331277,0,1613,2,"""CAD""",2021-02-20,2021-02-23


In [8]:
sales = sales.drop("Order Date","Delivery Date") 
# Remove the original string columns after converting them to date type.

In [9]:
sales = sales.rename({"CustomerKey" : "Customer Key",
                      "StoreKey" : "Store Key",
                      "ProductKey" : "Product Key",
                      "Order_Date" : "Order Date",
                      "Delivery_Date" : "Delivery Date"})
# Standardize column names.

In [10]:
stores = pl.read_csv(r"../data/raw/Stores.csv")
stores
# Read the second dataset.

StoreKey,Country,State,Square Meters,Open Date
i64,str,str,i64,str
1,"""Australia""","""Australian Capital Territory""",595,"""1/1/2008"""
2,"""Australia""","""Northern Territory""",665,"""1/12/2008"""
3,"""Australia""","""South Australia""",2000,"""1/7/2012"""
4,"""Australia""","""Tasmania""",2000,"""1/1/2010"""
5,"""Australia""","""Victoria""",2000,"""12/9/2015"""
…,…,…,…,…
63,"""United States""","""Utah""",2000,"""3/6/2008"""
64,"""United States""","""Washington DC""",1330,"""1/1/2010"""
65,"""United States""","""West Virginia""",1785,"""1/1/2012"""


In [11]:
stores.glimpse()

Rows: 67
Columns: 5
$ StoreKey      <i64> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
$ Country       <str> 'Australia', 'Australia', 'Australia', 'Australia', 'Australia', 'Australia', 'Canada', 'Canada', 'Canada', 'Canada'
$ State         <str> 'Australian Capital Territory', 'Northern Territory', 'South Australia', 'Tasmania', 'Victoria', 'Western Australia', 'New Brunswick', 'Newfoundland and Labrador', 'Northwest Territories', 'Nunavut'
$ Square Meters <i64> 595, 665, 2000, 2000, 2000, 2000, 1105, 2105, 1500, 1210
$ Open Date     <str> '1/1/2008', '1/12/2008', '1/7/2012', '1/1/2010', '12/9/2015', '1/1/2010', '5/7/2007', '7/2/2014', '3/4/2005', '4/4/2015'



In [12]:
stores.null_count().sum()
# One missing value was identified in Square Meters for the Online store.
# This is expected because an online store does not have a physical store area.
# The row was retained because the missing value is structurally meaningful.

StoreKey,Country,State,Square Meters,Open Date
u32,u32,u32,u32,u32
0,0,0,1,0


In [13]:
stores.is_duplicated().sum()
# No duplicate rows were found in the dataset.

0

In [14]:
stores = stores.with_columns(
    pl.col("Open Date").str.to_date("%m/%d/%Y").alias("Open_Date")
)
# Changed the data type of Order Date column to date.

In [15]:
stores = stores.drop("Open Date")

In [16]:
stores = stores.rename({"Open_Date" : "Open Date","StoreKey" : "Store Key"})

In [17]:
products = pl.read_csv(r"../data/raw/Products.csv")
# Read the third dataset

In [18]:
products.glimpse()

Rows: 2517
Columns: 10
$ ProductKey     <i64> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
$ Product Name   <str> 'Contoso 512MB MP3 Player E51 Silver', 'Contoso 512MB MP3 Player E51 Blue', 'Contoso 1G MP3 Player E100 White', 'Contoso 2G MP3 Player E200 Silver', 'Contoso 2G MP3 Player E200 Red', 'Contoso 2G MP3 Player E200 Black', 'Contoso 2G MP3 Player E200 Blue', 'Contoso 4G MP3 Player E400 Silver', 'Contoso 4G MP3 Player E400 Black', 'Contoso 4G MP3 Player E400 Green'
$ Brand          <str> 'Contoso', 'Contoso', 'Contoso', 'Contoso', 'Contoso', 'Contoso', 'Contoso', 'Contoso', 'Contoso', 'Contoso'
$ Color          <str> 'Silver', 'Blue', 'White', 'Silver', 'Red', 'Black', 'Blue', 'Silver', 'Black', 'Green'
$ Unit Cost USD  <str> '$6.62 ', '$6.62 ', '$7.40 ', '$11.00 ', '$11.00 ', '$11.00 ', '$11.00 ', '$30.58 ', '$30.58 ', '$30.58 '
$ Unit Price USD <str> '$12.99 ', '$12.99 ', '$14.52 ', '$21.57 ', '$21.57 ', '$21.57 ', '$21.57 ', '$59.99 ', '$59.99 ', '$59.99 '
$ SubcategoryKey <i64> 101, 101, 10

In [19]:
products.null_count()
# There are no null values in the dataset.

ProductKey,Product Name,Brand,Color,Unit Cost USD,Unit Price USD,SubcategoryKey,Subcategory,CategoryKey,Category
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0


In [20]:
products.is_duplicated().sum()
# No duplicate rows were found in the dataset.

0

In [21]:
products = products.rename(
    {"ProductKey" : "Product Key",
    "SubcategoryKey" : "Subcategory Key",
    "CategoryKey" : "Category Key"}
)
# Standardize column names.

In [22]:
products = products.with_columns(
    pl.col("Unit Price USD").str.replace("$","", literal = True).str.replace(",","").str.strip_chars().cast(pl.Float64),
    pl.col("Unit Cost USD").str.replace("$","", literal = True).str.replace(",","").str.strip_chars().cast(pl.Float64)
)

In [23]:
products = products.with_columns(
   (pl.col("Unit Price USD") - pl.col("Unit Cost USD")).alias("Gross Profit USD")
)

In [24]:
exchange_rates = pl.read_csv(r"../data/raw/Exchange_Rates.csv")

In [25]:
exchange_rates.null_count()
# There are no null values in the dataset.

Date,Currency,Exchange
u32,u32,u32
0,0,0


In [26]:
exchange_rates.is_duplicated().sum()

0

In [27]:
exchange_rates

Date,Currency,Exchange
str,str,f64
"""1/1/2015""","""USD""",1.0
"""1/1/2015""","""CAD""",1.1583
"""1/1/2015""","""AUD""",1.2214
"""1/1/2015""","""EUR""",0.8237
"""1/1/2015""","""GBP""",0.6415
…,…,…
"""2/20/2021""","""USD""",1.0
"""2/20/2021""","""CAD""",1.261
"""2/20/2021""","""AUD""",1.2723


In [28]:
exchange_rates = exchange_rates.with_columns(
    pl.col("Date").str.to_date("%m/%d/%Y").alias("Date_new")
)

In [29]:
exchange_rates = exchange_rates.drop("Date")
# Remove the previous Date column whose data type was str.

In [30]:
exchange_rates = exchange_rates.rename(
    {"Date_new" : "Date"}
)

In [31]:
data_dictionary = pl.read_csv(r"../data/raw/Data_Dictionary.csv")
# Dataset that contains general info about the columns of the separate tables and their content.

In [32]:
data_dictionary

Table,Field,Description
str,str,str
"""Sales""","""Order Number""","""Unique ID for each order"""
"""Sales""","""Line Item""","""Identifies individual products…"
"""Sales""","""Order Date""","""Date the order was placed"""
"""Sales""","""Delivery Date""","""Date the order was delivered"""
"""Sales""","""CustomerKey""","""Unique key identifying which c…"
…,…,…
"""Stores""","""Square Meters""","""Store footprint in square mete…"
"""Stores""","""Open Date""","""Store open date"""
"""Exchange Rates""","""Date""","""Date"""


In [33]:
data_dictionary.is_duplicated().sum()
# No duplicate records were found in the dataset.

0

In [34]:
data_dictionary.null_count().sum()
# No null values were found in the dataset.

Table,Field,Description
u32,u32,u32
0,0,0


In [35]:
customers = pl.read_csv(r"../data/raw/Customers.csv",
                        encoding = "cp1252", schema_overrides = {
                        "Zip Code" : pl.String}
                       )

In [36]:
customers

CustomerKey,Gender,Name,City,State Code,State,Zip Code,Country,Continent,Birthday
i64,str,str,str,str,str,str,str,str,str
301,"""Female""","""Lilly Harding""","""WANDEARAH EAST""","""SA""","""South Australia""","""5523""","""Australia""","""Australia""","""7/3/1939"""
325,"""Female""","""Madison Hull""","""MOUNT BUDD""","""WA""","""Western Australia""","""6522""","""Australia""","""Australia""","""9/27/1979"""
554,"""Female""","""Claire Ferres""","""WINJALLOK""","""VIC""","""Victoria""","""3380""","""Australia""","""Australia""","""5/26/1947"""
786,"""Male""","""Jai Poltpalingada""","""MIDDLE RIVER""","""SA""","""South Australia""","""5223""","""Australia""","""Australia""","""9/17/1957"""
1042,"""Male""","""Aidan Pankhurst""","""TAWONGA SOUTH""","""VIC""","""Victoria""","""3698""","""Australia""","""Australia""","""11/19/1965"""
…,…,…,…,…,…,…,…,…,…
2099600,"""Female""","""Denisa Dušková""","""Houston""","""TX""","""Texas""","""77017""","""United States""","""North America""","""3/25/1936"""
2099618,"""Male""","""Justin Solórzano""","""Mclean""","""VA""","""Virginia""","""22101""","""United States""","""North America""","""2/16/1992"""
2099758,"""Male""","""Svend Petrussen""","""Wilmington""","""NC""","""North Carolina""","""28405""","""United States""","""North America""","""11/9/1937"""


In [37]:
customers.glimpse()

Rows: 15266
Columns: 10
$ CustomerKey <i64> 301, 325, 554, 786, 1042, 1086, 1133, 1256, 1314, 1568
$ Gender      <str> 'Female', 'Female', 'Female', 'Male', 'Male', 'Male', 'Male', 'Male', 'Male', 'Male'
$ Name        <str> 'Lilly Harding', 'Madison Hull', 'Claire Ferres', 'Jai Poltpalingada', 'Aidan Pankhurst', 'Hayden Clegg', 'Nicholas Caffyn', 'Lincoln Jenks', 'Isaac Israel', 'Luke Virtue'
$ City        <str> 'WANDEARAH EAST', 'MOUNT BUDD', 'WINJALLOK', 'MIDDLE RIVER', 'TAWONGA SOUTH', 'TEMPLERS', 'JUBILEE POCKET', 'KULLOGUM', 'EDITH RIVER', 'KOTTA'
$ State Code  <str> 'SA', 'WA', 'VIC', 'SA', 'VIC', 'SA', 'QLD', 'QLD', 'NT', 'VIC'
$ State       <str> 'South Australia', 'Western Australia', 'Victoria', 'South Australia', 'Victoria', 'South Australia', 'Queensland', 'Queensland', 'Northern Territory', 'Victoria'
$ Zip Code    <str> '5523', '6522', '3380', '5223', '3698', '5371', '4802', '4660', '852', '3565'
$ Country     <str> 'Australia', 'Australia', 'Australia', 'Australia', 'Aus

In [38]:
customers.null_count().sum()
# No null values were found in the dataset.

CustomerKey,Gender,Name,City,State Code,State,Zip Code,Country,Continent,Birthday
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0


## Key Findings

- The Sales table contains 62,884 records and has no duplicate rows.
- Approximately 79.06% of Delivery Date values are missing.
- The missing Delivery Date values could not be reliably inferred from the available related tables, so they were retained as null.
- The Stores table contains one missing value in Square Meters for the Online store, which is structurally expected.
- The Products table contains no missing or duplicate records.
- Currency exchange rates are available for multiple currencies and dates, enabling currency-related analysis.
- Date fields were converted from strings to date types for reliable time-based analysis.